In [ ]:
import pandas as pd
import os

# ==========================
# FILE PATHS
# ==========================

files = [
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\Germany_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\India_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\Korea_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\Kunshan_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\SAME_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\SCEET_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\Tianjin_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\USA_ML_Dataset.xlsx",
r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\Cyclam_ML_Dataset.xlsx"
]

# dossier output
output_folder = r"C:\Users\INFOTEC\OneDrive\Bureau\ml_neww\clean_data"
os.makedirs(output_folder, exist_ok=True)

# ==========================
# COLONNES IMPORTANTES
# ==========================

columns_keep = [
"Date",
"Day_Index",
"Week",
"Week_of_Year",
"Country",
"Part_Number",
"Stock_Before",
"Stock_J1_Global",
"Coverage_Days",
"Perturb_Pct",
"Delta_Inter_Reappro",
"Usage_MA3",
"Daily_Usage_Mean",
"Usage_Trend",
"CV_Inter",
"Tendance_Globale",
"Safety_Stock",
"Reorder_Point",
"Risk_Score",
"Days_Before_Rupture",
"Risk_Value",
"ABC_Class",
"XYZ_Class",
"Rotation_Rate",
"Daily_Usage"
]

# ==========================
# TRAITEMENT
# ==========================

for file in files:

    print("Processing:", file)

    df = pd.read_excel(file)

    # تنظيف أسماء الأعمدة
    df.columns = df.columns.str.strip()

    # حذف الأعمدة غير الموجودة
    cols = [c for c in columns_keep if c in df.columns]
    df = df[cols]

    # تحويل التاريخ
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # ==========================
    # FEATURES TIME
    # ==========================

    df["Day_of_Week"] = df["Date"].dt.dayofweek
    df["Month"] = df["Date"].dt.month
    df["Is_Weekend"] = df["Day_of_Week"].isin([5,6]).astype(int)

    # ==========================
    # إصلاح الأرقام
    # ==========================

    df = df.replace(",", ".", regex=True)

    numeric_cols = df.columns.drop([
        "Country",
        "Part_Number",
        "ABC_Class",
        "XYZ_Class",
        "Usage_Trend",
        "Tendance_Globale"
    ], errors="ignore")

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ==========================
    # معالجة Daily_Usage = 0
    # ==========================

    df["Daily_Usage"] = df["Daily_Usage"].replace(0, pd.NA)

    df["Daily_Usage"] = df["Daily_Usage"].interpolate()

    # تعويض القيم الفارغة
    df = df.fillna(0)

    # ==========================
    # حفظ الملف الجديد
    # ==========================

    name = os.path.basename(file)

    output_path = os.path.join(output_folder, "clean_" + name)

    df.to_excel(output_path, index=False)

    print("Saved:", output_path)

print("All datasets cleaned successfully!")